In [0]:
# Run once - mounts all 4 containers to /mnt/petiot/
# If you run this in Databricks:
print(dbutils.secrets.get(scope="petiot", key="client-secret"))

# The output will strictly say:
# [REDACTED]

In [0]:
# This lists your keys safely without exposing the values
display(dbutils.secrets.list("petiot"))

# Depricated **method**

In [0]:
# # ==========================================
# # 1. AUTHENTICATION CONFIGURATION
# # ==========================================

# configs = {
#     "fs.azure.account.auth.type": "OAuth",
#     "fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
#     "fs.azure.account.oauth2.client.id": dbutils.secrets.get(scope="petiot", key="client-id"),
#     "fs.azure.account.oauth2.client.secret": dbutils.secrets.get(scope="petiot", key="client-secret"),
#     "fs.azure.account.oauth2.client.endpoint": f"https://login.microsoftonline.com/{dbutils.secrets.get(scope='petiot', key='tenant-id')}/oauth2/token"
# }

# # Define targets
# storage_account = "petiot"
# containers = ["raw", "bronze", "silver", "gold"]

# # ==========================================
# # 2. CLEAN UP BROKEN/EXISTING MOUNTS
# # ==========================================

# print("Checking for existing or stale mount points...")
# try:    current_mounts = [m.mountPoint for m in dbutils.fs.mounts()]
# except Exception:
#     current_mounts = []


# for container in containers:
#     mount_point = f"/mnt/petiot/{container}"
#     if mount_point in current_mounts:
#         print(f"-> Unmounting existing path to ensure fresh connection: {mount_point}")
#         try:
#             dbutils.fs.unmount(mount_point)
#         except Exception as e:
#             print(f"   Failed to unmount {mount_point}: {e}")

# print("-" * 50)

# # ==========================================
# # 3. EXECUTE FRESH MOUNTS
# # ==========================================
# print("Starting fresh mounting process...")
# for container in containers:
#     mount_point = f"/mnt/petiot/{container}"
#     source_url = f"abfss://{container}@{storage_account}.dfs.core.windows.net/"
    
#     try:
#         dbutils.fs.mount(
#             source=source_url,
#             mount_point=mount_point,
#             extra_configs=configs
#         )
#         print(f" Successfully Mounted: {mount_point}")
#     except Exception as e:
#         print(f"❌ Critical Failure mounting {mount_point}: {e}")

# print("-" * 50)

# # ==========================================
# # 4. VERIFY THE MOUNTS
# # ==========================================
# print("Verifying directory structures:")
# for container in containers:
#     path = f"/mnt/petiot/{container}"
#     try:
#         # Performs a lightweight directory listing to check actual network connectivity
#         files = dbutils.fs.ls(path)
#         print(f" Active and Readable: {path} (Contains {len(files)} items)")
#     except Exception as e:
#         print(f"❌ Unreadable Path: {path}. Check Azure RBAC 'Storage Blob Data Contributor' role permissions.")

##### DEPRECATED #####

# Direct Spark Session connection

In [0]:
# ==========================================
# 1. DIRECT SESSION AUTHENTICATION
# ==========================================
storage_account = "petiot"
containers = ["raw", "bronze", "silver", "gold","audit"]

print("Injecting credentials directly into active Spark Session Context...")

# Fetch your perfectly configured Key Vault secrets
client_id = dbutils.secrets.get(scope="petiot", key="client-id")
client_secret = dbutils.secrets.get(scope="petiot", key="client-secret")
tenant_id = dbutils.secrets.get(scope="petiot", key="tenant-id")

# Assign OAuth properties straight to the current cluster session
spark.conf.set(f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

print(" Setup complete! Storage connection authorized.")
print("-" * 60)

# ==========================================
# 2. RUN DIRECT CONNECTIVITY VERIFICATION
# ==========================================
print("Verifying secure container access:")
for container in containers:
    # We drop the /mnt/ prefix and call the actual Azure cloud URL directly
    direct_cloud_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net/"
    try:
        files = dbutils.fs.ls(direct_cloud_path)
        print(f" Successfully Connected: '{container}' container is fully accessible! (Found {len(files)} items)")
    except Exception as e:
        print(f"❌ Failed to reach '{container}' container: {e}")

# This one did work **😇**